# Surface Mesh Reconstruction

Occulus provides three surface reconstruction algorithms via Open3D:

| Method | Best for | Normals required? | Output |
|---|---|---|---|
| **Poisson** | Smooth watertight terrain/objects | Yes | Watertight |
| **Ball Pivoting (BPA)** | Thin-shell objects, open surfaces | Yes | Non-watertight |
| **Alpha Shape** | Convex/concave hulls | No | Non-watertight |

**Requires**: `pip install occulus[viz]`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from occulus.types import PointCloud
from occulus.normals import estimate_normals, orient_normals_to_viewpoint

try:
    from occulus.mesh import poisson_mesh, ball_pivoting_mesh, alpha_shape_mesh
    MESH_AVAILABLE = True
    print('Open3D mesh functions available.')
except ImportError:
    MESH_AVAILABLE = False
    print('open3d not installed — mesh cells will be skipped.')
    print('Install with: pip install occulus[viz]')

## 1. Create a Synthetic Terrain Point Cloud

In [ ]:
rng = np.random.default_rng(42)
n = 5_000

# Smooth rolling terrain patch (30m x 30m)
x = rng.uniform(0, 30, n)
y = rng.uniform(0, 30, n)
z = 3 * np.sin(x / 8) * np.cos(y / 10) + rng.normal(0, 0.03, n)

terrain = PointCloud(np.column_stack([x, y, z]).astype(np.float64))
print(f'Cloud: {terrain.n_points:,} points')

# Estimate normals oriented toward a viewpoint above
terrain_n = estimate_normals(terrain, radius=2.0)
terrain_n = orient_normals_to_viewpoint(terrain_n, viewpoint=np.array([15.0, 15.0, 100.0]))
print(f'Normals estimated. Up-facing: {(terrain_n.normals[:, 2] > 0).mean():.1%}')

## 2. Poisson Reconstruction

In [ ]:
if MESH_AVAILABLE:
    poisson = poisson_mesh(terrain_n, depth=7, density_threshold_quantile=0.05)
    print(f'Poisson: {poisson.n_vertices:,} vertices, {poisson.n_faces:,} faces')
else:
    print('Skipped (open3d not installed)')

## 3. Ball Pivoting Reconstruction

In [ ]:
if MESH_AVAILABLE:
    bpa = ball_pivoting_mesh(terrain_n, radii=[0.3, 0.6, 1.2])
    print(f'BPA   : {bpa.n_vertices:,} vertices, {bpa.n_faces:,} faces')
else:
    print('Skipped (open3d not installed)')

## 4. Alpha Shape

In [ ]:
if MESH_AVAILABLE:
    alpha = alpha_shape_mesh(terrain, alpha=1.5)
    print(f'Alpha : {alpha.n_vertices:,} vertices, {alpha.n_faces:,} faces')
else:
    print('Skipped (open3d not installed)')

## 5. Comparison Summary

In [ ]:
if MESH_AVAILABLE:
    print(f'{'Method':<12} {'Vertices':>10} {'Faces':>10} {'V/pt ratio':>12}')
    print('-' * 48)
    for name, mesh in [('Poisson', poisson), ('BPA', bpa), ('Alpha', alpha)]:
        print(f'{name:<12} {mesh.n_vertices:>10,} {mesh.n_faces:>10,} {mesh.n_vertices/n:>12.2f}')
    
    # Visual comparison of vertex density
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (name, mesh) in zip(axes, [('Poisson', poisson), ('BPA', bpa), ('Alpha', alpha)]):
        v = mesh.vertices
        ax.scatter(v[:, 0], v[:, 1], c=v[:, 2], cmap='terrain', s=0.5)
        ax.set_title(f'{name}\n{mesh.n_vertices:,} verts / {mesh.n_faces:,} faces', fontsize=10)
        ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    plt.suptitle('Surface Mesh Reconstruction Comparison', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../outputs/mesh_reconstruction.png', dpi=150)
    plt.show()

## 6. Save Mesh to PLY

In [ ]:
if MESH_AVAILABLE:
    import tempfile
    from pathlib import Path
    
    out_dir = Path('../../outputs')
    
    # Save via Open3D
    o3d_mesh = poisson.to_open3d()
    import open3d as o3d
    out_path = out_dir / 'terrain_poisson.ply'
    o3d.io.write_triangle_mesh(str(out_path), o3d_mesh)
    print(f'Saved: {out_path}')

**Next**: `08_geometric_features.ipynb`